In [5]:
import io
from googleapiclient.http import MediaIoBaseDownload
from docx import Document
from odf import text, teletype
from odf.opendocument import load
from PyPDF2 import PdfReader
import openpyxl
from pptx import Presentation

# -------------------------------------------------------
# FUNCIÓN PRINCIPAL: escanear Drive y extraer texto
# -------------------------------------------------------

def scan_drive_all(drive_service, batch_size=100):
    """
    Escanea todos los archivos de Drive compatibles y extrae texto.
    Retorna lista de diccionarios:
    [{"file_id": ..., "name": ..., "mime_type": ..., "text": ...}, ...]
    """

    # Mime types que vamos a escanear
    mime_types = [
        "application/pdf",
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",  # DOCX
        "application/vnd.oasis.opendocument.text",  # ODT
        "text/plain",  # TXT
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",  # XLSX
        "application/vnd.openxmlformats-officedocument.presentationml.presentation"  # PPTX
    ]

    archivos = []
    page_token = None

    while True:
        # Listar archivos
        query = " or ".join([f"mimeType='{mt}'" for mt in mime_types])
        results = drive_service.files().list(
            q=query,
            pageSize=batch_size,
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()

        files = results.get("files", [])
        if not files:
            break

        for f in files:
            file_id = f["id"]
            name = f["name"]
            mime_type = f["mimeType"]

            text_content = _download_and_extract_text(drive_service, file_id, mime_type)

            archivos.append({
                "file_id": file_id,
                "name": name,
                "mime_type": mime_type,
                "text": text_content
            })

        page_token = results.get("nextPageToken")
        if not page_token:
            break

    return archivos


# -------------------------------------------------------
# FUNCIONES AUXILIARES PARA EXTRAER TEXTO
# -------------------------------------------------------

def _download_and_extract_text(drive_service, file_id, mime_type):
    """
    Descarga archivo y extrae texto según tipo.
    """

    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False

    while not done:
        _, done = downloader.next_chunk()

    fh.seek(0)

    try:
        # PDF
        if mime_type == "application/pdf":
            reader = PdfReader(fh)
            return "\n".join([page.extract_text() or "" for page in reader.pages])

        # DOCX
        elif mime_type == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
            doc = Document(fh)
            return "\n".join([p.text for p in doc.paragraphs])

        # ODT
        elif mime_type == "application/vnd.oasis.opendocument.text":
            odt_doc = load(fh)
            return "\n".join([teletype.extractText(p) for p in odt_doc.getElementsByType(text.P)])

        # TXT
        elif mime_type == "text/plain":
            return fh.read().decode("utf-8", errors="ignore")

        # XLSX
        elif mime_type == "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet":
            wb = openpyxl.load_workbook(fh, data_only=True)
            all_text = []
            for sheet in wb.worksheets:
                for row in sheet.iter_rows(values_only=True):
                    all_text.append(" ".join([str(cell) for cell in row if cell is not None]))
            return "\n".join(all_text)

        # PPTX
        elif mime_type == "application/vnd.openxmlformats-officedocument.presentationml.presentation":
            prs = Presentation(fh)
            all_text = []
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        all_text.append(shape.text)
            return "\n".join(all_text)

        else:
            return ""

    except Exception as e:
        return ""
